<a href="https://colab.research.google.com/github/trainocate-japan/developing-agentic-ai-with-langchain/blob/main/chap03/exercise/starter/chap03_exercise_3B_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 演習 3-B【演習 (starter)】: ヘルプデスクエージェント v1 — ヘルプデスク Step 2

**研修コース「Agentic AI 開発実践 - LangChain 版」/ 第3章「エージェント開発の基本」**

この Notebook は演習 3-B の**問題 (starter)** です。
コード中の **`# TODO`** を埋めて、ヘルプデスクエージェント v1 を完成させてください。
TODO は **4 か所** (①〜④) です。TODO 以外のセルは完成状態なので、そこは実行するだけで構いません。

## この演習で作るもの

第2章 (Step 1) で**手動の Function Calling ループ**として実装したヘルプデスク QA を、
本章で学んだ `create_agent` で**書き直し**、さらに **FAQ 検索ツール (`search_faq`)** を追加した
「**ヘルプデスクエージェント v1**」を構築します。応答は後続のチケット管理システムに渡せるよう、
`SupportAnswer` (Pydantic) で**構造化**します。

本章のハンズオン 3-A で習得した部品 (`@tool` / `create_agent` / 軌跡の読解) に加え、
**構造化出力 (`response_format`)** をここで初めて手を動かして学びます。

完成すると、**FAQ 検索**と**稼働状況確認**の 2 ツールを持ち、
`category` / `answer` / `escalation_required` の構造化された回答を返すエージェントが手に入ります。

## ヘルプデスク演習ストーリーにおける位置づけ (Step 2)

本コースの演習は「**社内 IT ヘルプデスクエージェント**」を第2〜8章で段階的に拡張して完成させます。
この演習はその **Step 2** にあたります。

| 章 | 追加する要素 | 演習後の姿 |
|---|---|---|
| 第2章 | Function Calling 手動ループ | 稼働状況に答える素朴な QA ループ (openai 直接) |
| **第3章 (この演習)** | **create_agent / @tool / 構造化出力** | **FAQ 検索 + 稼働状況ツールを持つ単体エージェント** |
| 第4章以降 | メモリ / MCP / HITL / 評価 / マルチエージェント | … 最終的に Web UI から操作できるヘルプデスクへ |

> **第2章の成果物は不要です。** `get_system_status` の実装と FAQ データはこの Notebook に**配布済み**なので、
> 第2章の演習が未完了でも、この Notebook だけで完結します。

## 前提条件

- このファイルを **Google Colab** で開いていること
- Colab の **[シークレット]** に `OPENAI_API_KEY` を登録済みであること
  (第1章の演習 1-1 で登録済みのはずです。未登録でも、後述の「0. セットアップ」で登録できます)
- インターネット接続 (API を呼び出します)

## 所要時間

約 12 分

---
> **モデル名について**: モデル名は変数 `MODEL` に集約しています (例: `MODEL = "openai:gpt-5.4"`)。
> 研修実施時は講師が指定する最新モデル名に差し替えてください。


## 0. セットアップ

### 0-1. 依存パッケージのインストール

LangChain v1 本体 (`langchain`) と OpenAI 統合 (`langchain-openai`) をインストールします。

> 研修実施時は再現性のため、バージョンをピン留めすることを推奨します
> (本コースの基盤は **langchain 1.3.x / langchain-openai 1.3.x** です)。


In [ ]:
# LangChain v1 本体と OpenAI 統合を最新版へインストール/更新
# 研修実施時はバージョンをピン留め推奨 (langchain 1.3.x / langchain-openai 1.3.x)
!pip install -U langchain langchain-openai

### 0-2. API キーのセットアップ (Colab シークレット方式)

API キーは**コードに直接書かず**、Colab の **[シークレット]** 機能で管理します。

**操作手順** (未登録の場合):
1. 画面左の **鍵アイコン 🔑 [シークレット]** をクリック
2. **[新しいシークレットを追加]** を押す
3. 名前に `OPENAI_API_KEY`、値にあなたの API キーを入力
4. このノートブックからのアクセスを **オン** にする

次のセルは、Colab シークレットからキーを読み取り環境変数に設定します。
LangChain はこの環境変数を自動的に読みます。Colab 以外では、あらかじめ環境変数 `OPENAI_API_KEY` を
設定しておけば動きます。


In [ ]:
import os

# Colab のシークレットから API キーを読み込み、環境変数に設定する
# Colab 以外の環境では except 側に入り、既存の環境変数 OPENAI_API_KEY をそのまま使う
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Colab シークレットから OPENAI_API_KEY を読み込みました。")
except ImportError:
    # Colab 以外では環境変数 OPENAI_API_KEY が設定済みとみなす
    print("Colab 以外の環境です。環境変数 OPENAI_API_KEY を使用します。")

# モデル名は変数に集約 ("provider:model" 形式)。研修実施時に最新へ差し替え
MODEL = "openai:gpt-5.4"

print("APIキー設定済み:", bool(os.environ.get("OPENAI_API_KEY")))
print("使用モデル:", MODEL)

---

## 1. 配布コード — 稼働状況ツールと FAQ データ

ここは**配布済み**のコードです (実行するだけ)。エージェントに持たせる材料を 2 つ用意します。

- **`get_system_status(service)`** — 社内システムの稼働状況を返すダミーツール。
  第2章の演習で作ったものと同じ題材で、固定の辞書から状態を返します。**`@tool` で定義済み**です。
- **`FAQ_DATA`** — FAQ 検索ツールが参照する小さな FAQ データ (dict)。VPN・パスワード・経費精算などを数件収録しています。

`get_system_status` は `@tool` でのツール定義の**お手本**にもなっています
(docstring と型ヒントの書き方を、次の TODO① の参考にしてください)。


In [ ]:
from langchain.tools import tool

# --- 配布: 社内システムの稼働状況を返すダミーデータ (第2章と整合) ---
SYSTEM_STATUS = {
    "勤怠システム": "正常稼働中",
    "経費精算システム": "正常稼働中",
    "メールサーバー": "一部遅延あり (調査中)",
    "VPN": "メンテナンス中 (本日 22:00 まで)",
}


# --- 配布: 稼働状況ツール (@tool 定義のお手本) ---
@tool
def get_system_status(service: str) -> str:
    """指定された社内システムの現在の稼働状況を取得する。

    システムが「動いているか」「障害が出ていないか」「メンテナンス中か」といった
    稼働状態の問い合わせに使う。

    Args:
        service: 稼働状況を知りたい社内システムの名称 (例: 勤怠システム, VPN)
    """
    status = SYSTEM_STATUS.get(service, "不明 (登録されていないシステムです)")
    return f"{service}の稼働状況: {status}"


# --- 配布: FAQ 検索ツールが参照する FAQ データ ---
FAQ_DATA = {
    "VPN": "VPN に接続できない場合は、まず社内ポータルから最新の VPN クライアントを再インストールし、"
           "二要素認証アプリの時刻同期を確認してください。それでも解決しない場合は情報システム部へ。",
    "パスワード": "パスワードを忘れた場合は、社内ポータルの『パスワード再設定』から手続きできます。"
                  "ロックされた場合の解除は、本人確認のうえ情報システム部での対応が必要です。",
    "経費精算": "経費精算は経費精算システムから申請します。領収書は PDF で添付し、月末締め翌月 10 日払いです。",
    "メール": "メールの容量超過時は、添付ファイルの大きい古いメールを削除するか、アーカイブを利用してください。",
}


# 動作確認
print(get_system_status.invoke({"service": "勤怠システム"}))
print("FAQ キーワード:", list(FAQ_DATA.keys()))

---

## 2. FAQ 検索ツールを定義する — search_faq【TODO①】

`search_faq(keyword)` を `@tool` で定義します。配布の `FAQ_DATA` をキーワードで検索して
該当する FAQ 本文を返すツールです。関数の中身は記述済みです。

**あなたのタスク (TODO①)**: `search_faq` の **docstring** と **型ヒント** を書いてください。

> **ヒント**: docstring は**モデルへの指示文**です (ハンズオン 3-4 で実演したとおり)。
> 「**何を・いつ使うか**」を書きます。たとえば「社内 FAQ をキーワードで検索し、よくある質問への回答を返す。
> VPN・パスワード・経費精算などの『手続き・やり方』の問い合わせに使う」のように、
> このツールがいつ選ばれるべきかをモデルが判断できるように書くのがコツです。
> 配布済みの `get_system_status` の docstring が良いお手本です。
> 型ヒントは `keyword: str` と戻り値 `-> str` を付けます (型ヒントは入力スキーマの生成に必須)。


In [ ]:
@tool
def search_faq(keyword: str) -> str:  # TODO①: 型ヒント (keyword: str, 戻り値 -> str) は記述済み。docstring を埋める
    # TODO①: ここに docstring を書く (モデルへの指示文。「何を・いつ使うか」)
    #         例:「社内 FAQ をキーワードで検索し、よくある質問への回答を返す。
    #              VPN・パスワード・経費精算などの手続き・やり方の問い合わせに使う。」
    #         配布済みの get_system_status の docstring を参考にしてください。

    # ↓ 関数の中身は記述済み。FAQ_DATA をキーワードで検索する
    hits = [text for key, text in FAQ_DATA.items() if keyword in key or key in keyword]
    if hits:
        return "\n".join(hits)
    return "該当する FAQ が見つかりませんでした。"


# 動作確認
print(search_faq.invoke({"keyword": "VPN"}))
print("---")
print("description:", search_faq.description)

---

## 3. 構造化出力スキーマを定義する — SupportAnswer【TODO③】

エージェントの応答を「機械が扱えるデータ」として受け取るため、出力スキーマを **Pydantic モデル**で宣言します。
後続のチケット管理システムが処理できるよう、3 つのフィールドを持たせます。

- `category` (str): 問い合わせの分類 (例: "VPN", "パスワード", "稼働状況")
- `answer` (str): ユーザーへの回答本文
- `escalation_required` (bool): 情報システム部へのエスカレーションが必要か

**あなたのタスク (TODO③ の前半)**: 各 `Field` に **description** を書いてください。
`Field(description=...)` の description は、3-4 の docstring と同じく**モデルへの指示文**として働きます。
各フィールドに何を入れてほしいかを明確に伝えます。

> **ヒント**: import は `from pydantic import BaseModel, Field` です。
> description には「この欄に何を入れるか」を書きます。たとえば `escalation_required` なら
> 「本人確認やアカウント操作など、情報システム部の対応が必要なら True」のように、
> True/False の判断基準まで書くとモデルが安定します。


In [ ]:
from pydantic import BaseModel, Field


class SupportAnswer(BaseModel):
    """ヘルプデスクの一次対応の回答。チケット管理システムに渡せる構造化データ。"""

    # TODO③ (前半): 各 Field の description を埋める (モデルへの指示文)
    category: str = Field(
        description=""  # TODO③: 例「問い合わせの分類 (例: VPN, パスワード, 稼働状況)」
    )
    answer: str = Field(
        description=""  # TODO③: 例「ユーザーに提示する回答本文。簡潔な一次対応の案内」
    )
    escalation_required: bool = Field(
        description=""  # TODO③: 例「情報システム部への引き継ぎが必要なら True」
    )


print("SupportAnswer スキーマを定義しました:", list(SupportAnswer.model_fields.keys()))

---

## 4. エージェントを構成する — create_agent【TODO②・TODO③後半】

いよいよ `create_agent` で「ヘルプデスクエージェント v1」を組み立てます。
ハンズオン 3-3 の天気エージェントと同じ要領ですが、ツールが 2 つになり、構造化出力が加わります。

**あなたのタスク**:
- **TODO②**: `model` / `tools` / `system_prompt` を指定する
  - `model` には `MODEL` を渡す
  - `tools` には FAQ 検索と稼働状況確認の**2 つ** (`search_faq`, `get_system_status`) をリストで渡す
  - `system_prompt` は「**社内 IT ヘルプデスクの一次対応担当**」としての役割を書く
- **TODO③ (後半)**: `response_format` に `SupportAnswer` を指定する

> **ヒント**: `system_prompt` は「あなたは社内 IT ヘルプデスクの一次対応担当です。…」のように役割と方針を書きます
> (例:「FAQ で答えられることは FAQ 検索ツールを、システムの稼働状況はステータス確認ツールを使って調べる。
> 一次対応で完結しない場合はエスカレーションを案内する」)。
> 構造化出力は `result["messages"]` ではない**別のキー**に入ります (次のセルで取り出します)。
> `response_format=SupportAnswer` と渡すだけで、LangChain がモデルの能力に応じて最適な戦略を自動選択します。


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    # TODO②: model / tools / system_prompt を指定する
    model=...,            # TODO②: MODEL を渡す
    tools=...,            # TODO②: [search_faq, get_system_status] の 2 ツールをリストで渡す
    system_prompt=...,    # TODO②: 「社内 IT ヘルプデスクの一次対応担当」としての役割を書く
    # TODO③(後半): response_format に SupportAnswer を指定する (構造化出力の宣言)
)

print("ヘルプデスクエージェント v1 を構成しました。")
print("戻り値の型:", type(agent).__name__)   # CompiledStateGraph

---

## 5. 実行して構造化された回答を取り出す【TODO④】

「**VPN に繋がらないんだけど**」と相談してエージェントを実行します。
構造化出力は `result["messages"]` (会話の軌跡) ではなく、エージェント最終状態の
**`structured_response` キー**に入ります。ここが本演習の最重要ポイントです。

**あなたのタスク (TODO④)**: `result` から構造化された回答 (`SupportAnswer` インスタンス) を取り出してください。

> **ヒント**: 構造化出力は `result["structured_response"]` に入っています。
> 取り出したオブジェクトは検証済みの Pydantic インスタンスなので、
> `answer.category` / `answer.answer` / `answer.escalation_required` のように**属性アクセス**できます。


In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "VPN に繋がらないんだけど"}]}
)

# TODO④: 構造化出力を取り出す。result["messages"] ではない別のキーに入っている
answer = ...   # TODO④: result["structured_response"] を代入する

print("型              :", type(answer).__name__)        # SupportAnswer
print("category        :", answer.category)              # 属性アクセスできる
print("answer          :", answer.answer)
print("escalation_required:", answer.escalation_required)

---

## 6. 軌跡を読む — どのツールがどの順で呼ばれたか

ここは**完成済み**のセルです (TODO ではありません)。ハンズオン 3-3 で身につけた軌跡読解のスキルで、
`result["messages"]` を読みます。「VPN に繋がらない」に対して、どのツールが呼ばれたかを確認しましょう。

**期待される結果**: おおむね `HumanMessage → AIMessage(tool_calls=search_faq) → ToolMessage → ...`
という並びになり、FAQ 検索ツールが呼ばれていることが読み取れます
(最後に構造化出力用の往復が入るため、末尾のメッセージはやや増えることがあります)。

構造化出力が **`structured_response`** に、会話の軌跡が **`messages`** に、それぞれ分かれて入っている点を
あらためて確認してください。


In [ ]:
# 実行軌跡をダンプする (どのツールがどの順で呼ばれたかを読む)
for m in result["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    called = [tc["name"] for tc in tool_calls] if tool_calls else None
    # content は長くなりがちなので先頭 60 文字だけ表示
    content_preview = (str(m.content)[:60] + "…") if m.content else ""
    print(f"{type(m).__name__:14} | tool_calls={called} | {content_preview}")

### (おまけ) 稼働状況の質問も試す

別のツール (`get_system_status`) が選ばれることも確認しておきましょう。
「**勤怠システムは動いていますか?**」と聞くと、今度は稼働状況ツールが呼ばれるはずです。
これも完成済みのセルです。


In [ ]:
result2 = agent.invoke(
    {"messages": [{"role": "user", "content": "勤怠システムは動いていますか?"}]}
)
ans2 = result2["structured_response"]
print("category           :", ans2.category)
print("answer             :", ans2.answer)
print("escalation_required:", ans2.escalation_required)
print("--- 軌跡 (呼ばれたツール) ---")
for m in result2["messages"]:
    tool_calls = getattr(m, "tool_calls", None)
    if tool_calls:
        print("  tool_calls:", [tc["name"] for tc in tool_calls])

---

## まとめ — 完成の目安

このエージェント v1 で、本章の学習目標 3〜5 を総動員しました。

| 使った部品 | 学んだ節 | この演習での役割 |
|---|---|---|
| `@tool` (docstring + 型ヒント) | 3-4 | `search_faq` の定義。docstring がツール選択を左右する |
| `create_agent` (model/tools/system_prompt) | 3-3 | 2 ツールを束ねるヘルプデスクエージェント本体 |
| 軌跡 `result["messages"]` の読解 | 3-3 | どのツールがどの順で呼ばれたかを読む |
| `response_format` (Pydantic) | 3-5 | `SupportAnswer` で構造化。`structured_response` から取得 |

### 完成の目安 (達成できたか確認)

- ✅ 「VPN に繋がらないんだけど」で実行すると `result["structured_response"]` から
  `category` / `answer` / `escalation_required` が取り出せる (TODO①〜④)
- ✅ 「勤怠システムは動いていますか?」では `get_system_status` が呼ばれる (ツールの呼び分け)
- ✅ 軌跡 `result["messages"]` を ReAct ループ (Human → AI(tool_calls) → Tool → AI) に対応付けて読める

4 つの TODO を埋め終えたら、上から順にすべて実行して、完成の目安を満たすか確認してください。

> **構造化出力の置き場所**、もう一度だけ強調します。
> 会話の軌跡は `result["messages"]`、検証済みの構造化データは `result["structured_response"]`。
> この 2 つは別のキーです。後続システムに渡すのは後者です。

### 次章の予告

このエージェント v1 は、まだ問い合わせのたびに会話を忘れます (ハンズオン 3-3 のステートレス実験を思い出してください)。
第4章では `checkpointer` を 1 つ加えて、社員ごとに会話を記憶し、トレースで診断できる **v2** へ拡張します。
